# Chest X-Ray Pneumonia Detection Report

This notebook is the final local report for the trained ResNet50 pneumonia classifier. It uses the saved checkpoint, metrics, and generated figures already stored in the repository.

**Headline result:** test accuracy is **90.4%** across **624** images, with **PR-AUC 0.975** and **ROC-AUC 0.965** for the Pneumonia class.

**Clinical-use disclaimer:** this project is for education and research only. It is not validated or approved for clinical diagnosis.


## 1. Local Setup

This report does not train the model or call Colab. It reads local artifacts from `models/` and `reports/figures/`.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
CHECKPOINT = MODELS_DIR / "best_resnet50.pth"
METRICS_PATH = FIGURES_DIR / "test_metrics.json"

required = [CHECKPOINT, METRICS_PATH]
missing = [str(path.relative_to(PROJECT_ROOT)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required report artifacts: {missing}")

with METRICS_PATH.open() as f:
    metrics = json.load(f)

print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint: {CHECKPOINT.relative_to(PROJECT_ROOT)} ({CHECKPOINT.stat().st_size / 1024**2:.1f} MB)")
print(f"Metrics: {METRICS_PATH.relative_to(PROJECT_ROOT)}")


## 2. Model and Training Summary

The final model is a ResNet50 transfer-learning classifier with a two-class output: `NORMAL` and `PNEUMONIA`. Training used ImageNet-pretrained features, class-imbalance handling, and validation F1 to select the best checkpoint.


In [ ]:
def show_image(path, title=None, figsize=(10, 6)):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path.relative_to(PROJECT_ROOT)}")
        return
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(plt.imread(path))
    ax.axis("off")
    ax.set_title(title or path.name.replace("_", " ").replace(".png", "").title())
    plt.show()

show_image(FIGURES_DIR / "training_curves.png", "Training and Validation Curves", figsize=(14, 5))


## 3. Test Performance

The test split contains **624** chest X-rays. The model performs better on Pneumonia recall than Normal recall, which is expected when optimizing for a medical screening style task where missed pneumonia cases are especially important.


In [ ]:
report = metrics["classification_report"]
rows = []
for label in ["NORMAL", "PNEUMONIA"]:
    rows.append({
        "Class": label,
        "Precision": report[label]["precision"],
        "Recall": report[label]["recall"],
        "F1-score": report[label]["f1-score"],
        "Support": int(report[label]["support"]),
    })

summary_df = pd.DataFrame(rows)
summary_df.loc[len(summary_df)] = {
    "Class": "Weighted avg",
    "Precision": report["weighted avg"]["precision"],
    "Recall": report["weighted avg"]["recall"],
    "F1-score": report["weighted avg"]["f1-score"],
    "Support": int(report["weighted avg"]["support"]),
}

display(summary_df.style.format({
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1-score": "{:.3f}",
}))

print(f"Accuracy: {report['accuracy']:.4f}")
print(f"PR-AUC:   {metrics['pr_auc']:.4f}")
print(f"ROC-AUC:  {metrics['roc_auc']:.4f}")


In [ ]:
for filename in ["confusion_matrix.png", "precision_recall_curve.png", "roc_curve.png"]:
    show_image(FIGURES_DIR / filename, figsize=(8, 6))


### Performance Interpretation

From the confusion matrix:

- **True Normal:** 186
- **False Positive:** 48 Normal images classified as Pneumonia
- **False Negative:** 12 Pneumonia images classified as Normal
- **True Pneumonia:** 378

The model catches most pneumonia cases with **96.9% Pneumonia recall**, missing **12** pneumonia cases out of **390**. The main weakness is lower **Normal recall (79.5%)**, meaning the model sometimes flags Normal images as Pneumonia.


## 4. Explainability: Grad-CAM and Eigen-CAM

Each comparison shows the original X-ray, Grad-CAM overlay, and Eigen-CAM overlay. These visualizations help check whether the model is focusing on plausible lung-region evidence rather than irrelevant image borders or labels.


In [ ]:
cam_dir = FIGURES_DIR / "cam_comparisons"
cam_images = sorted(cam_dir.glob("*.png")) if cam_dir.exists() else []
print(f"CAM comparisons found: {len(cam_images)}")

for img_path in cam_images[:8]:
    show_image(img_path, title=img_path.name, figsize=(15, 4))


## 5. Failure Case Analysis

Failure cases are the most important examples to inspect because they show where the model is likely to be unreliable. False negatives are more clinically concerning because they represent missed pneumonia cases; false positives increase follow-up burden but are usually safer in a screening context.


In [ ]:
failure_dir = FIGURES_DIR / "failure_cases"
failure_images = sorted(failure_dir.glob("*.png")) if failure_dir.exists() else []
print(f"Failure cases found: {len(failure_images)}")

for img_path in failure_images:
    show_image(img_path, title=img_path.stem.replace("_", " ").title(), figsize=(15, 4))


## 6. Key Insights and Limitations

### Key Insights

- The model reaches **90.4% accuracy** and **90.2% weighted F1**, which is strong for a binary pneumonia screening baseline.
- Pneumonia detection is the strongest behavior: **96.9% recall** and **92.6% F1**.
- The model produces more false alarms than missed pneumonia cases: **48 false positives** versus **12 false negatives**.
- The high **PR-AUC (0.975)** indicates strong ranking quality for the positive Pneumonia class under class imbalance.
- CAM visualizations provide a useful sanity check, but they are explanations of model attention, not proof of medical correctness.

### Limitations

- The dataset is pediatric and may not generalize to adult patients or other hospitals.
- The task is binary only: it does not distinguish viral from bacterial pneumonia.
- There is no external validation on another dataset such as NIH ChestX-ray14, CheXpert, or MIMIC-CXR.
- The model is not calibrated for clinical decision thresholds.
- This system is not approved for clinical use.

### Recommended Next Steps

- Validate on an external dataset before making stronger claims.
- Calibrate probabilities and choose thresholds based on the intended screening tradeoff.
- Add lung segmentation or bounding-box checks to verify CAM attention is anatomically plausible.
